In [36]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [37]:
df = pd.read_csv(r"phase3_ship_mode_prediction_dataset.csv")

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.drop(columns=['country.1', 'year', 'month', 'day', 'dayofweek', 'isweekend'])
df.columns

Index(['order_id', 'customer_id', 'order_priority', 'market', 'sales',
       'quantity', 'discount', 'profit', 'shipping_cost', 'ship_mode',
       'region', 'country', 'city', 'isreturned'],
      dtype='object')

In [38]:
df

,order_id,customer_id,order_priority,market,sales,quantity,discount,profit,shipping_cost,ship_mode,region,country,city,isreturned
0,AE-2011-9160,PO-8865,Medium,EMEA,78.407997,6,0.7,-88.991997,3.870000,Standard Class,emea,united arab emirates,ajman,No
1,AE-2011-9160,PO-8865,Medium,EMEA,82.674004,2,0.7,-157.085999,5.690000,Standard Class,emea,united arab emirates,ajman,No
2,AE-2013-1130,EB-4110,High,EMEA,4.248000,1,0.7,-4.692000,0.100000,Same Day,emea,united arab emirates,ras al khaymah,No
3,AE-2013-1130,EB-4110,High,EMEA,224.748001,6,0.7,-232.272003,60.080002,Same Day,emea,united arab emirates,ras al khaymah,No
4,AE-2013-1530,MY-7380,High,EMEA,6.966000,1,0.7,-8.604000,1.750000,Second Class,emea,united arab emirates,ras al khaymah,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,IT-2012-5715811,GH-14485,Medium,EU,379.808990,3,0.1,-38.061001,39.209999,Standard Class,south,spain,san sebastian,No
29996,IT-2012-5715811,GH-14485,Medium,EU,535.653015,3,0.1,232.082993,31.530001,Standard Class,south,spain,san sebastian,No
29997,IT-2012-5750622,CC-12685,Medium,EU,11.310000,2,0.5,-9.330000,0.380000,Standard Class,central,france,toulouse,No
29998,IT-2012-5750622,CC-12685,Medium,EU,14.430000,2,0.5,-11.850000,1.380000,Standard Class,central,france,toulouse,No


In [39]:
print('our data info: ')
print(df.shape)
print("\n")
print("\n")
print('our shipping mode: ')
print(df["ship_mode"].value_counts())

our data info: 
(30000, 14)




our shipping mode: 
ship_mode
Standard Class    17996
Second Class       6053
First Class        4414
Same Day           1537
Name: count, dtype: int64


In [40]:
outliers_sales = df[df['sales'] < 0]
outliers_discount = df[(df['discount'] < 0) | (df['discount'] > 1)]
outliers_shipping = df[df['shipping_cost'] < 0]

print("outliers sales:")
print(outliers_sales.shape[0])
print("outliers discount:")
print(outliers_discount.shape[0])
print("outliers shipping:")
print(outliers_shipping.shape[0])


high_sales = df[df['sales'] > df['sales'].quantile(0.99)]
print("Top 1% high sales orders:")
print(high_sales.head(10))


outliers sales:
0
outliers discount:
0
outliers shipping:
0
Top 1% high sales orders:
            order_id customer_id order_priority market        sales  quantity  \
564     BO-2014-8960     MZ-7335           High   EMEA  2757.780029         6   
725   CA-2011-102988    GM-14695           High     US  4164.049805         5   
1069  CA-2011-116246    LW-17215         Medium     US  3785.290039         6   
1085  CA-2011-116904    SC-20095         Medium     US  9449.950195         5   
1102  CA-2011-117639    MW-18235           High     US  2715.929932         7   
1134  CA-2011-119375    YC-21895           High     US  2934.330078         7   
1149  CA-2011-120474    RP-19390           High     US  2807.840088         8   
1342  CA-2011-127299    JL-15835           High     US  2624.989990         3   
1376  CA-2011-128209    GT-14710         Medium     US  4007.840088        10   
1639  CA-2011-139892    BM-11140         Medium     US  8159.950195         8   

      discount       p

In [41]:
y = df['ship_mode']
X = df.drop(columns=['ship_mode', 'order_id', 'customer_id', 'city', 'profit', 'shipping_cost'])

num_cols = ['sales', 'quantity', 'discount']

cat_cols = []
for col in X.columns:
    if col not in num_cols:
        if df[col].nunique() < 15:
            cat_cols.append(col)


encoder = OneHotEncoder(drop="first", sparse_output=False)

X_encoded = pd.DataFrame(encoder.fit_transform(X[cat_cols]))
X_encoded.columns = encoder.get_feature_names_out(cat_cols)

scaler = StandardScaler()
num_data_scaled = pd.DataFrame(scaler.fit_transform(X[num_cols]), columns=num_cols)

num_data = X[num_cols].reset_index(drop=True)
encoded_data = X_encoded

X_final = pd.concat([num_data_scaled.reset_index(drop=True), X_encoded.reset_index(drop=True)], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)


X_final


,sales,quantity,discount,order_priority_High,order_priority_Low,order_priority_Medium,market_Africa,market_Canada,market_EMEA,market_EU,...,region_central asia,region_east,region_emea,region_north,region_north asia,region_oceania,region_south,region_southeast asia,region_west,isreturned_Yes
0,-0.360396,1.058552,3.359549,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.352575,-0.712491,3.359549,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.496369,-1.155251,3.359549,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.092083,1.058552,3.359549,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.491385,-1.155251,3.359549,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,0.192221,-0.269730,-0.083900,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
29996,0.477960,-0.269730,-0.083900,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
29997,-0.483420,-0.712491,2.211733,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29998,-0.477700,-0.712491,2.211733,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [43]:
y_pred = model.predict(X_test)

In [44]:
print('accurate model: ')
print(accuracy_score(y_test, y_pred))
print('\n')
print('\n')
print('all report:')
print(classification_report(y_test, y_pred))

accurate model: 
0.5295




all report:
                precision    recall  f1-score   support

   First Class       0.24      0.23      0.23       875
      Same Day       0.11      0.10      0.10       308
  Second Class       0.26      0.24      0.25      1189
Standard Class       0.70      0.73      0.72      3628

      accuracy                           0.53      6000
     macro avg       0.33      0.32      0.33      6000
  weighted avg       0.51      0.53      0.52      6000



In [45]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)



y_pred_xgb_encoded = xgb_model.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print('accurate model: ')
print(accuracy_xgb * 100)
print('\n')
print('\n')
print('all report:')
print(classification_report(y_test, y_pred_xgb))

accurate model: 
62.866666666666674




all report:
                precision    recall  f1-score   support

   First Class       0.41      0.17      0.24       875
      Same Day       0.23      0.02      0.04       308
  Second Class       0.32      0.10      0.15      1189
Standard Class       0.67      0.96      0.79      3628

      accuracy                           0.63      6000
     macro avg       0.41      0.31      0.30      6000
  weighted avg       0.54      0.63      0.54      6000



In [46]:
df['sales_per_quantity'] = df['sales'] / df['quantity'].replace(0, 1e-6)  


import numpy as np
df['discount_level'] = np.where(df['discount'] > 0.5, 'high',
                               np.where(df['discount'] > 0.1, 'medium', 'low'))


priority_map = {'High': 3, 'Medium': 2, 'Low': 1}
df['order_priority_num'] = df['order_priority'].map(priority_map).fillna(1)


new_num_cols = ['sales_per_quantity', 'order_priority_num']
num_cols.extend(new_num_cols)


cat_cols.append('discount_level')


X = df.drop(columns=['ship_mode', 'order_id', 'customer_id', 'city', 'profit', 'shipping_cost'])

encoder = OneHotEncoder(drop="first", sparse_output=False)
X_encoded = pd.DataFrame(encoder.fit_transform(X[cat_cols]))
X_encoded.columns = encoder.get_feature_names_out(cat_cols)

scaler = StandardScaler()
num_data_scaled = pd.DataFrame(scaler.fit_transform(X[num_cols]), columns=num_cols)

X_final_new = pd.concat([num_data_scaled.reset_index(drop=True), X_encoded.reset_index(drop=True)], axis=1)

X_train_new, X_test_new, y_train, y_test = train_test_split(X_final_new, y, test_size=0.2, random_state=42)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

gb_model = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.1, max_depth=3)
gb_model.fit(X_train_new, y_train_encoded)

y_pred_encoded = gb_model.predict(X_test_new)
y_pred = le.inverse_transform(y_pred_encoded)

print('accurate model: ')
print(accuracy_score(y_test, y_pred) * 100)
print('\n')
print('\n')
print('all report:')
print(classification_report(y_test, y_pred))

accurate model: 
63.916666666666664




all report:
                precision    recall  f1-score   support

   First Class       0.45      0.19      0.27       875
      Same Day       0.00      0.00      0.00       308
  Second Class       0.42      0.03      0.06      1189
Standard Class       0.66      1.00      0.79      3628

      accuracy                           0.64      6000
     macro avg       0.38      0.31      0.28      6000
  weighted avg       0.55      0.64      0.53      6000



c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [57]:
order_ids = []
actual_modes = []
predicted_modes = []

for i in range(len(y_test)):
    order_id = df['order_id'][X_test_new.index[i]]
    actual_mode = y_test.iloc[i]
    predicted_mode = y_pred[i]
    
    order_ids.append(order_id)
    actual_modes.append(actual_mode)
    predicted_modes.append(predicted_mode)

predictions = pd.DataFrame({
    'Order_ID': order_ids,
    'Actual_Ship_Mode': actual_modes,
    'Predicted_Ship_Mode': predicted_modes
})


predictions.to_csv('predictions_Ship_Mode.csv', index=False)


report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.index.name = 'Class'
report_df.to_csv('model_classification_report_Ship_Mode.csv', index=True)

params = gb_model.get_params()


model_df = pd.DataFrame({
    'Model': ['XGBoost'],
    'Accuracy': [accuracy_score(y_test, y_pred) * 100],
    'n_estimators': [params['n_estimators']],
    'learning_rate': [params['learning_rate']],
    'max_depth': [params['max_depth']],
})


model_df.to_csv('model_accuracy_with_params.csv', index=False)

c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag